In [1]:
import requests
import json
from pathlib import Path
from datetime import datetime, timezone

OPEN_METEO_URL = "https://api.open-meteo.com/v1/forecast"

WEATHER_LOCATIONS = [
    {"location_id": "london", "location_name": "London", "latitude": 51.5072, "longitude": -0.1276},
    {"location_id": "manchester", "location_name": "Manchester", "latitude": 53.4808, "longitude": -2.2426},
    {"location_id": "birmingham", "location_name": "Birmingham", "latitude": 52.4862, "longitude": -1.8904},
    {"location_id": "edinburgh", "location_name": "Edinburgh", "latitude": 55.9533, "longitude": -3.1883},
    {"location_id": "cardiff", "location_name": "Cardiff", "latitude": 51.4816, "longitude": -3.1791},
]

HOURLY_VARIABLES = [
    "temperature_2m",
    "precipitation",
    "wind_speed_10m",
    "relative_humidity_2m",
]


In [2]:

def validate_weather_payload(payload: dict) -> None:
    hourly = payload.get("hourly")

    if not hourly:
        raise ValueError("API response does not contain hourly data")

    times = hourly.get("time")
    temperatures = hourly.get("temperature_2m")
    precipitation = hourly.get("precipitation")

    if times is None:
        raise ValueError("API response does not contain hourly time values")

    if temperatures is None:
        raise ValueError("API response does not contain temperature values")

    if precipitation is None:
        raise ValueError("API response does not contain precipitation values")

    if not (
        len(times) == len(temperatures) == len(precipitation)
    ):
        raise ValueError("Weather arrays have different lengths")

In [3]:
def extract_weather_for_locations() -> dict:
    params = {
        "latitude": ",".join(
            str(location["latitude"]) for location in WEATHER_LOCATIONS
        ),
        "longitude": ",".join(
            str(location["longitude"]) for location in WEATHER_LOCATIONS
        ),
        "hourly": ",".join(HOURLY_VARIABLES),
        "timezone": "Europe/London",
        "past_days": 7,
    }

    response = requests.get(
        OPEN_METEO_URL,
        params=params,
        timeout=30,
    )
    response.raise_for_status()

    payload = response.json()

    if not isinstance(payload, list):
        raise ValueError("Expected a list of location responses from Open-Meteo")

    if len(payload) != len(WEATHER_LOCATIONS):
        raise ValueError(
            f"Expected {len(WEATHER_LOCATIONS)} location responses, got {len(payload)}"
        )

    for location, location_payload in zip(WEATHER_LOCATIONS, payload):
        location_payload["location_id"] = location["location_id"]
        location_payload["location_name"] = location["location_name"]

        validate_weather_payload(location_payload)
        
    return payload

In [4]:
payload = extract_weather_for_locations()

In [5]:
type(payload)

list

In [10]:
body = json.dumps(payload) #.encode("utf-8")
type(body)

str

In [9]:
json_lines = "\n".join(
                json.dumps(location, ensure_ascii=False)
                for location in payload
        ) + "\n"
type(json_lines)

str

In [11]:
def upload_json(data: dict) -> None: #upload_json(s3_client, bucket_name: str, key: str, data: dict) -> None:

    body = json.dumps(data).encode("utf-8")

    # s3_client.put_object(
    #     Bucket=bucket_name,
    #     Key=key,
    #     Body=body,
    #     ContentType="application/json",
    # )

    return body


In [12]:
def upload_json_2(data: dict) -> None: #upload_json_2(s3_client, bucket_name: str, key: str, data: dict, format: str) -> None:

    json_lines = "\n".join(
        json.dumps(location, ensure_ascii=False)
        for location in data
    ) + "\n"

        # s3_client.put_object(
        #     Bucket=BUCKET_NAME,
        #     Key=key,
        #     Body=json_lines.encode("utf-8"),
        #     ContentType="application/x-ndjson",
        # )
    
    return json_lines

In [13]:
body_f = upload_json(payload)

In [14]:
json_lines_f = upload_json_2(payload)

In [15]:
json_lines_f

'{"latitude": 51.51147, "longitude": -0.13078308, "generationtime_ms": 0.6864070892333984, "utc_offset_seconds": 0, "timezone": "Europe/London", "timezone_abbreviation": "GMT", "elevation": 16.0, "hourly_units": {"time": "iso8601", "temperature_2m": "°C", "precipitation": "mm", "wind_speed_10m": "km/h", "relative_humidity_2m": "%"}, "hourly": {"time": ["2026-11-06T00:00", "2026-11-06T01:00", "2026-11-06T02:00", "2026-11-06T03:00", "2026-11-06T04:00", "2026-11-06T05:00", "2026-11-06T06:00", "2026-11-06T07:00", "2026-11-06T08:00", "2026-11-06T09:00", "2026-11-06T10:00", "2026-11-06T11:00", "2026-11-06T12:00", "2026-11-06T13:00", "2026-11-06T14:00", "2026-11-06T15:00", "2026-11-06T16:00", "2026-11-06T17:00", "2026-11-06T18:00", "2026-11-06T19:00", "2026-11-06T20:00", "2026-11-06T21:00", "2026-11-06T22:00", "2026-11-06T23:00", "2026-11-07T00:00", "2026-11-07T01:00", "2026-11-07T02:00", "2026-11-07T03:00", "2026-11-07T04:00", "2026-11-07T05:00", "2026-11-07T06:00", "2026-11-07T07:00", "2026